# Modules and Device

In [ ]:
%pip install nltk
%pip install evaluate
%pip install sentencepiece
%pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b43c04f1a6417402802320eb0dff6249b4febf62582236b757e5eb4ef08d1a86
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
# Base --------------------------------------------------
import pandas as pd

# PyTorch -----------------------------------------------
import torch

# NLP ---------------------------------------------------
import nltk
from nltk.tokenize import sent_tokenize

# Transformers ------------------------------------------
from transformers import pipeline, set_seed
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
#import sentencepiece

# Hugging Face -----------------------------------------
# from huggingface_hub import login, list_datasets, notebook_login   # Use this if you want to push your models to HuggingFace
from datasets import load_dataset

# Evaluation --------------------------------------------
from rouge_score import rouge_scorer, scoring
import evaluate

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Data Set

We are going to use the CNN/DailyMail v3.0.0 data set from HuggingFace which is a nonanonymized set containing 300K pairs of news articles and their corresponding summaries which is made from the bullet points that CNN and DailyMail attach to their articles

In [ ]:
dataset = load_dataset("cnn_dailymail", '3.0.0')
print(f"Features: {dataset['train'].column_names}")

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Features: ['article', 'highlights', 'id']


We see that the dataset has three columns:

 1. The article column contains the news articles.
 2. The highlights column contains the summaries.
 3. The ids column uniquely identifies each article.

let's see the first of them in the training set

In [ ]:
sample = dataset["train"][0]
print(f"""Article (excerpt of 500 characters, total length: {len(sample["article"])}): """)
print(sample["article"][:500])
print(f'\nSummary (length: {len(sample["highlights"])}):')
print(sample["highlights"])

Article (excerpt of 500 characters, total length: 2527): 
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s

Summary (length: 217):
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


As you see the text can be rather long (2527) compared to the target summary (281). And here we find the first challenge for transformers since the context size is usually limited to 1000 tokens.

The standard way of dealing with this is truncate the texts beyond the model's context size...which is a problem since there may be important information beyond this limit!

# Text Summarization

Text summarization is about abstraction, not extraction, i.e. we do not want to retrieve text, but to generate an abstract from the content which would be new sentences and not excerpts from the text.

For this task we will first explore the result of the application of some of the most "famous" models and we will restrict ourselves to a maximum of 2000 characters to have the same input in all the models.

A common convection in text summarization is to separate the summary sentences by a new line. We may add the newline token, but the NLTK (Natural Language Tookkit) package includes more sofisticated algorithms for this which can differentiate the end of a sentence from a punctuation in an abbreviation

In [ ]:
nltk.download("punkt_tab")    # note that older versions of NLTK had this as "punkt"

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Let's see how this works

In [ ]:
s = "The U.S.A. is a country. The U.N. is an orgatnization."
sent_tokenize(s)

['The U.S.A. is a country.', 'The U.N. is an orgatnization.']

So, as required, we have two separated sentences.

A common baseline for summarization are the first three sentences of the article. We can define a simple function to retrieve them as follows

In [ ]:
def three_sentence_summary(text):
    return '\n'.join(sent_tokenize(text)[:3])

We will collect all the summaries of each model in a dictionary, then

In [ ]:
sample_text = dataset['train'][1]['article'][:2000]     # just selecting this article to generate summaries from each model

summaries = {}
summaries['baseline'] = three_sentence_summary(sample_text)

### GPT-2

Now that we have the baseline, we can start appliying our models. Let's start with the GPT2 model that we already used to generate text.

A surprising feature of the model is that we can generate summaries with it by simply appending "TL;DR" at the end of the input text! This expression means "Too long, didn't read" and is used in sites as Reddit to indicate a short version of a long post...

We will create a language generation pipeline that predicts the words that will follow a specified prompt

In [ ]:
pipe = pipeline('text-generation', model='gpt2-xl', device=device)

set_seed(42)
gpt2_query = sample_text + "\nTL;DR:\n"
pipe_out = pipe(gpt2_query, max_length=512, clean_up_tokenization_spaces=True)
summaries["gpt2"] = "\n".join(sent_tokenize(pipe_out[0]["generated_text"][len(gpt2_query) :]))

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### T5

T5 is a universal transformer achitecture in which all the tasks are text-to-text tasks. Here the checkpoitns are trained on a mixture of unsupervised data, to reconstruct masked words, and supervised data for several tasks.

We can directly load t5 for summarization with the `pipeline()` function.

In [ ]:
pipe = pipeline('text-generation', model='google-t5/t5-large', device=device)
pipe_out = pipe(sample_text)
summaries["t5"] = "\n".join(sent_tokenize(pipe_out[0]["generated_text"]))

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

### BART

Finally, BART, is also an encoder-decoder architecture trained to reconstruct corrupted inputs. It contains pretrained schemes of BERT and GPT-2. We will use a fine tuned checkpoint created for the CNN/DailyMail data

In [ ]:
model_id = "facebook/bart-large-cnn"

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

inputs = tok(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(device)

summary_ids = model.generate(
    **inputs,
    max_new_tokens=120,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

summary_text = tok.decode(summary_ids[0], skip_special_tokens=True)
summaries["bart"] = "\n".join(sent_tokenize(summary_text))

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

### PEGASUS

This is another encoder-decoder architecture with the pretraining objective to predict masked sentences in multisentenced texts.

In [ ]:
model_id = "google/pegasus-cnn_dailymail"

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

inputs = tok(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(device)

summary_ids = model.generate(
    **inputs,
    max_new_tokens=120,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

summary = tok.decode(summary_ids[0], skip_special_tokens=True)
summaries["pegasus"] = "\n".join(sent_tokenize(summary))

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Measuring the quality

Is there a way to decide about the quality of the text generated in the summaries?

In [ ]:
print("GROUND TRUTH")
print(dataset["train"][1]["highlights"])
print("")

for model_name in summaries:
    print(model_name.upper())
    print(summaries[model_name])
    print("")

GROUND TRUTH
Mentally ill inmates in Miami are housed on the "forgotten floor"
Judge Steven Leifman says most are there as a result of "avoidable felonies"
While CNN tours facility, patient shouts: "I am the son of the president"
Leifman says the system is unjust and he's fighting for change .

BASELINE
Editor's note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events.
Here, Soledad O'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial.
MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor."

GPT2
The mentally ill aren't being treated properly and are being housed in an environment that is not conducive to their care.
The inmates are living on the ninth floor of the jail, a place where there are no beds and no p

From the texts above we may somehow decide which text works best for us and the first thing we see is that GPT-2 is significantly different from the others (why?)... it can sometimes hallucinate a bit with the texts. However, we may better need some formal metric to decide of the correctness of the text. For this, there are two common metrics used to evaluate the generated text: BLEU and ROUGE

### BLEU

BLEU (Bilingual Evaluation Understudy) is a method that was developed in 2002 and published as "BLEU: a method for automatic evaluation of machine translations", K. Papineni et al. with the following idea: instead of looking at how many tokens in the generate text are aligned with the reference text tokens, we look at words or n-grams.

This is a precision metric, which means that we look at how many of the n-grams in the generated text are present in the reference. It's strict because even if the generated text misses large parts of the reference but has some correct n-grams, it will score based on those correct parts.

It is computed as

$$
BLEU = min(1, e^{1-l_{ref}/l_{gen}})\left(\prod_{i=1}^N p_n\right)^{1/N}
$$

where $p_n$ is the precision of the n-grams, and the first factor is known as the brevity penalty, since the precision would otherwise favor the generation of short sequences.

Take a look at [this](https://medium.com/towards-data-science/evaluating-text-output-in-nlp-bleu-at-your-own-risk-e8609665a213) to fully understand this metric, but some key observations are that BLEU does not consider meaning or sentence structure, it cannot handle morphologically complex languages, and does not map well to human judgements. So... maybe we need another metric!

### ROUGE

Rouge is a metric based on recall (remember the difference between recall and precision?). The aproach, otherwise, is similar to BLEU: we look for different n-grams and compare their occurrences in the generated text and ompare with the reference text. The difference is that with ROUGE we check how many n-grams in the reference text also occur in the generated text, then it rewards models for covering as much of the reference content as possible.

The metrics we receive are the F-scores of four different measures

 * ROUGE-1 (Unigram Overlap):
    * It measures the overlap of unigrams (individual words) between the generated and the reference text.
    * It captures basic content similarity at the word level, then it is used as a general measure of content overlap
    * Formula

$$
ROUGE_1 = \frac{\text{Number of overlapping unigrams}}{\text{Total unigrams in reference (for recall)}}
$$

  * Example: <br>
  Reference: "The cat sat on the mat"<br>
  Candidate: "The cat is on the mat"<br>
  Overlap: {The, cat, on, the, mat} $\to$ 5 overlapping unigrams<br>
  ROUGE-1 (Recall): $\frac{5}{6}$

<br>

 * ROUGE_2 (Bigram Overlap)
    * It measures the overlap of bigrams (two consecutive words) between generate and reference text.
    * It captures fluency and more contextual accuracy than ROUGE-1, then it evaluates not just content but also sequence coherence
    * Formula:

$$
ROUGE_2 = \frac{\text{Number of overlapping bigrams}}{\text{Total bigrams in reference}}
$$

 * Example: <br>
  Reference: "The cat sat on the mat" $\to$ bigrams: {The cat, cat sat, sat on, on the, the mat}<br>
  Candidate: "The cat is on the mat" $\to$ bigrams: {The cat, cat is, is on, on the, the mat}<br>
  Overlap: {The cat, on the, the mat} $\to$ 3 overlapping bigrams<br>
  ROUGE-2 (Recall): $\frac{3}{5}$

<br>

* ROUGE-L (Longest Common Subsequence - LCS)
    * It measures the length of the longest common subsequence (LCS) between the candidate and reference texts.
    * It captures fluency and sentence-level structure without requiring strict consecutive matches. It is good for assessing longer sequences, maintaining flexibility with word order
    * Formula:
    
$$
ROUGE_L = \frac{\text{LCS length}}{\text{Reference length}}
$$

  * Example: <br>
  Reference: "The cat sat on the mat"<br>
  Candidate: "The cat is on the mat"<br>
  LCS: "The cat on the mat" → length = 5<br>
  ROUGE-L (Recall): $\frac{5}{6}$

<br>

* ROUGE-Lsum (Summary-Level LCS)
    * It is a variant of ROUGE-L designed specifically for multi-sentence summaries. It calculates LCS across the entire summary rather than sentence-by-sentence.
    * It measures overall coherence and structure in summarization tasks. Then it is preferred for document summarization tasks, especially when comparing long texts
    * The Key Difference from ROUGE-L is that while ROUGE-L compares sentence-wise LCS and aggregates scores, ROUGE-Lsum treats the summary as one long sequence for LCS computation.


In [ ]:
rouge_metric = evaluate.load("rouge")


reference = dataset["train"][1]["highlights"]
records = []
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

for model_name in summaries:
    rouge_metric.add(prediction=summaries[model_name], reference=reference)
    score = rouge_metric.compute()
    rouge_dict = dict((rn, score[rn]) for rn in rouge_names)
    records.append(rouge_dict)
pd.DataFrame.from_records(records, index=summaries.keys())

,rouge1,rouge2,rougeL,rougeLsum
baseline,0.365079,0.145161,0.206349,0.285714
gpt2,0.159420,0.036496,0.130435,0.159420
t5,0.188295,0.081841,0.122137,0.173028
bart,0.475248,0.222222,0.316832,0.415842
pegasus,0.000000,0.000000,0.000000,0.000000


How can we interpret these metrics?

As a summary of our previous definitions, let's remember that thee ROUGE scores are comparing overlap between the generated summaries and a reference summary at different granularities:

* ROUGE-1 → unigram overlap (content words)
* ROUGE-2 → bigram overlap (local fluency / phrase matching)
* ROUGE-L → longest common subsequence (structure similarity)
* ROUGE-Lsum → sentence-level structural overlap (often used for summarization)

Now, model by model:

1. Baseline: This is a moderately strong baseline. It captures a reasonable amount of content words (ROUGE-1) but has weaker phrase-level coherence (ROUGE-2). Structure alignment (ROUGE-L) is limited.
2. GPT-2: Underperforms the baseline in unigram and bigram overlap. The very low ROUGE-2 suggests weak alignment at phrase level. However, ROUGE-L and ROUGE-Lsum are slightly higher than the baseline, meaning that it sometimes captures longer structural sequences even if exact phrase overlap is low.
3. T5: Improves unigram overlap over both baseline and GPT-2. Bigram overlap is slightly below the baseline but much higher than GPT-2. ROUGE-L and especially ROUGE-Lsum are clearly better, suggesting stronger structural alignment with the reference.
4. BART: Clearly dominates across all metrics. This suggests BART is best at both selecting relevant content and reproducing reference-like phrasing and structure. The large jump in ROUGE-2 (0.222) is particularly meaningful: it is matching meaningful chunks, not just isolated words.
5. Pegasus: ROUGE-1 is relatively low (similar to GPT-2), but ROUGE-2 is high (second only to BART). That means it matches important phrases even if overall word overlap is lower. ROUGE-L and Lsum are solid but below BART and T5.
This pattern often indicates more aggressive abstraction: fewer total overlapping words, but when it overlaps, it overlaps meaningfully.

The interpretation at a conceptual level:

 * ROUGE-1 measures coverage.
 * ROUGE-2 measures coherence and faithfulness to phrasing.
 * ROUGE-L measures structural similarity.

Then we see that BART is great at all three. T5 is balanced and robust. Pegasus appears more abstractive. GPT-2, being primarily a language model and not fine-tuned specifically for summarization (unless explicitly done), behaves less optimally here. All of which (just a final comment) align well with what we may expect from the architecture of the models themselves.